# 09 — Population Analysis & Bonding: Charges, Bond Orders & NBO

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ppt-2/Ch121a-DFT/blob/main/notebooks/09_nbo_population_analysis.ipynb)

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:
- Explain Mulliken and Löwdin population analysis and their limitations
- Compute atomic partial charges with PySCF
- Understand the Wiberg bond order and interpret bond order matrices
- Describe Natural Population Analysis (NPA) and Natural Bond Orbitals (NBO)
- Recognize the basis-set dependence of Mulliken charges
- Visualise charge distributions and compare across molecules
- Connect population analysis to Lewis structure and hyperconjugation concepts

## 1. Theory: Population Analysis

### 1.1 Mulliken Population Analysis

Given the density matrix $\mathbf{P}$ and overlap matrix $\mathbf{S}$, the **Mulliken population** on atom $A$ is:

$$q_A^{\text{Mulliken}} = Z_A - \sum_{\mu \in A} (\mathbf{PS})_{\mu\mu}$$

where $Z_A$ is the nuclear charge and the sum runs over basis functions centred on $A$. This partitions overlap density equally between the two atoms — an arbitrary choice that makes Mulliken charges **strongly basis-set dependent**.

### 1.2 Löwdin Population Analysis

**Symmetric (Löwdin) orthogonalisation** transforms the basis:

$$|\tilde{\mu}\rangle = \sum_\nu (\mathbf{S}^{-1/2})_{\nu\mu} |\nu\rangle$$

The Löwdin charges use the density matrix in this orthogonal basis:

$$q_A^{\text{Löwdin}} = Z_A - \sum_{\mu \in A} \tilde{P}_{\mu\mu}, \quad \tilde{\mathbf{P}} = \mathbf{S}^{1/2}\mathbf{P}\mathbf{S}^{1/2}$$

Löwdin charges are less basis-set sensitive than Mulliken charges but still not unique.

### 1.3 Wiberg Bond Index

The **Wiberg bond order** between atoms $A$ and $B$ measures the covalent bond multiplicity:

$$B_{AB} = \sum_{\mu \in A} \sum_{\nu \in B} (\mathbf{PS})_{\mu\nu}(\mathbf{PS})_{\nu\mu}$$

| $B_{AB}$ | Interpretation |
|:--------:|----------------|
| ~1.0 | Single bond |
| ~1.5 | Aromatic / resonance |
| ~2.0 | Double bond |
| ~3.0 | Triple bond |

### 1.4 Natural Population Analysis (NPA) & NBO

**Natural Bond Orbitals** (Weinhold & Landis) provide a rigorous basis-set-independent partitioning:

1. Build **natural atomic orbitals** (NAOs) by block-diagonalising $\mathbf{P}$ in the atom-centred subspace
2. Identify **natural hybrids** (NHOs) from atom–atom block diagonalisation
3. Combine NHOs to form **NBOs**: one-centre lone pairs $n_A$ and two-centre bonds $\sigma_{AB}$
4. NPA charges: $q_A^{\text{NPA}} = Z_A - \sum_{\mu \in A} n_\mu^{\text{NAO}}$

NBO **donor–acceptor interaction energies** (hyperconjugation):

$$E^{(2)}_{\text{donor}\to\text{acceptor}} = -n_\sigma \frac{\langle \sigma | \hat{F} | \sigma^* \rangle^2}{\varepsilon_{\sigma^*} - \varepsilon_\sigma}$$

where $n_\sigma$ is the donor occupancy and $\varepsilon$ are NBO energies.

In [ ]:
%%time
# =============================================================================
# Ch121a: Quantum Chemistry & DFT — Notebook 09: Population Analysis
# License: GPL-3.0 (https://www.gnu.org/licenses/gpl-3.0.en.html)
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import matplotlib.pyplot as plt
from pyscf import gto, dft, scf
from pyscf.lo import orth

def run_rks(atom_str, basis='def2-SVP', xc='B3LYP', verbose=0):
    mol = gto.Mole(atom=atom_str, basis=basis, verbose=verbose)
    mol.build()
    mf = dft.RKS(mol)
    mf.xc = xc
    mf.verbose = verbose
    mf.kernel()
    return mol, mf

def mulliken_charges(mol, mf):
    """Compute Mulliken charges from density and overlap matrices."""
    dm   = mf.make_rdm1()            # AO density matrix
    S    = mol.intor('int1e_ovlp')   # overlap matrix
    PS   = dm @ S                    # P * S product
    pop  = np.einsum('ii->i', PS)    # diagonal: population per AO
    # Sum over AOs on each atom
    charges = np.array([mol.atom_charge(i) for i in range(mol.natm)], dtype=float)
    for i, (ao_start, ao_end) in enumerate(zip(*mol.aoslice_by_atom()[:, 2:].T)):
        charges[i] -= pop[ao_start:ao_end].sum()
    # Re-do using pyscf built-in for clarity
    pop_mull, chg_mull = mf.mulliken_pop(verbose=0)
    return chg_mull

def lowdin_charges(mol, mf):
    """Compute Löwdin charges via symmetric orthogonalisation."""
    dm = mf.make_rdm1()
    S  = mol.intor('int1e_ovlp')
    # S^(1/2) via eigen-decomposition
    eigvals, eigvecs = np.linalg.eigh(S)
    S_half = eigvecs @ np.diag(np.sqrt(eigvals)) @ eigvecs.T
    # Löwdin density matrix: P_tilde = S^(1/2) P S^(1/2)
    P_tilde = S_half @ dm @ S_half
    pop_lowdin = np.einsum('ii->i', P_tilde)
    charges = np.array([mol.atom_charge(i) for i in range(mol.natm)], dtype=float)
    atom_slices = mol.aoslice_by_atom()
    for i in range(mol.natm):
        ao_s, ao_e = atom_slices[i, 2], atom_slices[i, 3]
        charges[i] -= pop_lowdin[ao_s:ao_e].sum()
    return charges

# ------------------------------------------------------------------
# H2O charges
# ------------------------------------------------------------------
water_geom = 'O 0 0 0.117; H 0  0.757 -0.469; H 0 -0.757 -0.469'
mol_h2o, mf_h2o = run_rks(water_geom)

mull_h2o  = mulliken_charges(mol_h2o, mf_h2o)
lowdin_h2o = lowdin_charges(mol_h2o, mf_h2o)

print('Water (H₂O) — B3LYP/def2-SVP')
print('=' * 50)
print(f'  Atom   Mulliken   Löwdin')
for i in range(mol_h2o.natm):
    sym = mol_h2o.atom_symbol(i)
    print(f'  {sym:5s}   {mull_h2o[i]:+.4f}     {lowdin_h2o[i]:+.4f}')
print(f'  Sum    {sum(mull_h2o):+.4f}     {sum(lowdin_h2o):+.4f}  (should be 0)')

In [ ]:
%%time
# ------------------------------------------------------------------
# Charges for NH3, BH3, BF3 — comparison across molecules
# ------------------------------------------------------------------

molecules = {
    'NH₃': 'N 0 0 0.116; H 0  0.939 -0.269; H  0.813 -0.469 -0.269; H -0.813 -0.469 -0.269',
    'BH₃': 'B 0 0 0; H 1.189 0 0; H -0.595  1.030 0; H -0.595 -1.030 0',
    'BF₃': 'B 0 0 0; F 1.307 0 0; F -0.654  1.132 0; F -0.654 -1.132 0',
}

results = []
for mol_name, geom in molecules.items():
    mol, mf = run_rks(geom)
    mull  = mulliken_charges(mol, mf)
    lowdin = lowdin_charges(mol, mf)
    # Record central atom and first substituent charges
    results.append({
        'Molecule': mol_name,
        'Central atom': mol.atom_symbol(0),
        'Mulliken (central)': round(mull[0], 3),
        'Löwdin (central)': round(lowdin[0], 3),
        'Mulliken (H/F)': round(mull[1], 3),
        'Löwdin (H/F)': round(lowdin[1], 3),
    })
    print(f'{mol_name}:')
    for i in range(mol.natm):
        print(f'  {mol.atom_symbol(i):4s}  Mulliken {mull[i]:+.3f}   Löwdin {lowdin[i]:+.3f}')
    print()

df_charges = pd.DataFrame(results)
print('Summary:')
print(df_charges.to_string(index=False))

In [ ]:
%%time
# ------------------------------------------------------------------
# Wiberg Bond Orders
# ------------------------------------------------------------------
def wiberg_bond_orders(mol, mf):
    """Compute Wiberg (Mayer) bond orders in the Löwdin-orthogonalised basis."""
    dm = mf.make_rdm1()
    S  = mol.intor('int1e_ovlp')
    eigvals, eigvecs = np.linalg.eigh(S)
    S_half = eigvecs @ np.diag(np.sqrt(eigvals)) @ eigvecs.T
    # PS product in orthogonal basis
    PS = S_half @ dm @ S_half
    # Bond order matrix: B_AB = sum_{mu in A, nu in B} (PS)_{mu,nu}^2
    atom_slices = mol.aoslice_by_atom()
    natm = mol.natm
    BO = np.zeros((natm, natm))
    for A in range(natm):
        sA, eA = atom_slices[A, 2], atom_slices[A, 3]
        for B in range(natm):
            sB, eB = atom_slices[B, 2], atom_slices[B, 3]
            BO[A, B] = np.sum(PS[sA:eA, sB:eB] ** 2)
    return BO

print('Wiberg Bond Orders (Löwdin basis) — B3LYP/def2-SVP')
print('=' * 55)

for mol_name, geom in [('H₂O', water_geom),
                        ('NH₃', molecules['NH₃']),
                        ('BH₃', molecules['BH₃']),
                        ('BF₃', molecules['BF₃'])]:
    if mol_name == 'H₂O':
        mol, mf = mol_h2o, mf_h2o
    else:
        mol, mf = run_rks(molecules[mol_name])
    BO = wiberg_bond_orders(mol, mf)
    print(f'\n{mol_name}:')
    # Print unique off-diagonal bond orders
    seen = set()
    for A in range(mol.natm):
        for B in range(A+1, mol.natm):
            pair = (mol.atom_symbol(A), mol.atom_symbol(B))
            print(f'  {mol.atom_symbol(A)}–{mol.atom_symbol(B)} bond order: {BO[A,B]:.3f}')

print()
print('Expected approximate values:')
print('  O–H in H₂O  : ~0.80 (polar bond lowers covalent order)')
print('  N–H in NH₃  : ~0.85')
print('  B–H in BH₃  : ~0.88')
print('  B–F in BF₃  : ~0.55–0.70 (F lone pairs and partial π character)')

In [ ]:
# ------------------------------------------------------------------
# Basis-Set Dependence of Mulliken Charges for H2O
# ------------------------------------------------------------------
bases = ['STO-3G', '3-21G', '6-31G*', 'def2-SVP', 'def2-TZVP', 'def2-QZVP']
o_charges_mull  = []
o_charges_lowdin = []

for basis in bases:
    try:
        mol, mf = run_rks(water_geom, basis=basis)
        mull   = mulliken_charges(mol, mf)
        lowdin = lowdin_charges(mol, mf)
        o_charges_mull.append(mull[0])
        o_charges_lowdin.append(lowdin[0])
    except Exception as e:
        o_charges_mull.append(float('nan'))
        o_charges_lowdin.append(float('nan'))

print('Basis-set dependence of O charge in H₂O (B3LYP):')
print(f"  {'Basis':12s}  {'Mulliken (O)':>14s}  {'Löwdin (O)':>12s}")
for b, m, l in zip(bases, o_charges_mull, o_charges_lowdin):
    print(f'  {b:12s}  {m:+14.4f}  {l:+12.4f}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(bases, o_charges_mull,  'o-', color='steelblue', lw=2, label='Mulliken')
ax.plot(bases, o_charges_lowdin,'s--', color='coral',    lw=2, label='Löwdin')
ax.set_xlabel('Basis Set', fontsize=11)
ax.set_ylabel('Charge on O', fontsize=11)
ax.set_title('Basis-Set Dependence of Atomic Charges in H₂O\n(B3LYP, values relative to neutral O)', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', lw=0.8, ls='--')
plt.tight_layout()
plt.show()

print('\nKey observation:')
print('Mulliken charges vary wildly with basis set (especially large bases).')
print('Löwdin charges are more stable but still basis-dependent.')
print('NPA charges (from NBO analysis) are the most basis-set independent.')

In [ ]:
# ------------------------------------------------------------------
# Charge Comparison Bar Chart across H2O, NH3, BH3, BF3
# ------------------------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np

mol_names = ['H₂O', 'NH₃', 'BH₃', 'BF₃']
central_atoms = ['O', 'N', 'B', 'B']

# Collect central-atom Mulliken and Löwdin charges
mull_central  = []
low_central   = []
mull_ligand   = []
low_ligand    = []

for mol_name, geom in [('H₂O', water_geom), ('NH₃', molecules['NH₃']),
                        ('BH₃', molecules['BH₃']), ('BF₃', molecules['BF₃'])]:
    if mol_name == 'H₂O':
        mol, mf = mol_h2o, mf_h2o
    else:
        mol, mf = run_rks(molecules[mol_name])
    mull   = mulliken_charges(mol, mf)
    lowdin = lowdin_charges(mol, mf)
    mull_central.append(mull[0])
    low_central.append(lowdin[0])
    mull_ligand.append(mull[1])
    low_ligand.append(lowdin[1])

x = np.arange(len(mol_names))
width = 0.2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Central atom
bars1 = ax1.bar(x - width/2, mull_central, width, label='Mulliken', color='steelblue', alpha=0.85)
bars2 = ax1.bar(x + width/2, low_central,  width, label='Löwdin',   color='coral',     alpha=0.85)
ax1.set_xticks(x); ax1.set_xticklabels([f'{n}\n({a})' for n,a in zip(mol_names, central_atoms)])
ax1.set_ylabel('Atomic Charge (e)')
ax1.set_title('Central Atom Charges')
ax1.legend(); ax1.axhline(0, color='black', lw=0.8); ax1.grid(True, alpha=0.3, axis='y')

# Ligand (H or F)
ligand_names = ['H', 'H', 'H', 'F']
bars3 = ax2.bar(x - width/2, mull_ligand, width, label='Mulliken', color='steelblue', alpha=0.85)
bars4 = ax2.bar(x + width/2, low_ligand,  width, label='Löwdin',   color='coral',     alpha=0.85)
ax2.set_xticks(x); ax2.set_xticklabels([f'{n}\n({l})' for n,l in zip(mol_names, ligand_names)])
ax2.set_ylabel('Atomic Charge (e)')
ax2.set_title('Ligand (H or F) Charges')
ax2.legend(); ax2.axhline(0, color='black', lw=0.8); ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle('Mulliken vs Löwdin Atomic Charges — B3LYP/def2-SVP', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('Trends to note:')
print('  H₂O: O is strongly negative (high electronegativity)')
print('  NH₃: N is negative (lone pair donor); H slightly positive')
print('  BH₃: B slightly positive; H slightly negative (H more electronegative than expected)')
print('  BF₃: B strongly positive; F strongly negative (high electronegativity of F)')

## 2. Natural Bond Orbital (NBO) Analysis — Overview

PySCF provides localised orbital tools in `pyscf.lo`. Full NBO analysis (as in the Weinhold NBO program) requires an external interface, but natural atomic orbitals (NAOs) and intrinsic bond orbitals (IBOs) can be constructed:

```python
from pyscf.lo import orth, nao

# Natural Atomic Orbitals for H2O
mol, mf = run_rks('O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469')

# NAO coefficients (C_nao) — unitary rotation of AO basis
C_nao = nao.nao_nr(mol, mf)   # shape: (nao, nmo)

# NPA charges from NAO occupancies
dm_nao = C_nao.T @ mf.make_rdm1() @ C_nao  # density matrix in NAO basis
nao_occ = np.diag(dm_nao)                   # diagonal = occupancy

# NPA charge on each atom
npa_charges = np.array([mol.atom_charge(i) for i in range(mol.natm)], dtype=float)
for i, (s, e) in enumerate(zip(*mol.aoslice_by_atom()[:, 2:].T)):
    npa_charges[i] -= nao_occ[s:e].sum()
```

**Donor–acceptor hyperconjugation** (requires full NBO program):
- In ethane: $\sigma_{\text{CH}} \to \sigma^*_{\text{CH}}$ interactions stabilise the staggered conformation
- In vinyl fluoride: F lone pair $\to \pi^*_{\text{C=C}}$ delocalisation shortens the C–F bond
- In BF₃: empty B $p_z \to$ F lone pair back-donation gives partial $\pi$ character to B–F

These interactions are quantified by $E^{(2)}$ energies, available from NBO 6/7 or JANPA.

## 🔬 Research Connection

Population analysis is ubiquitous in computational chemistry:

- **Reactivity prediction**: Atoms with high NPA charges are nucleophilic or electrophilic reaction sites. Fukui functions (HOMO/LUMO density) refine this.
- **Force field development**: RESP (Restrained Electrostatic Potential) charges are fit to reproduce the QM electrostatic potential — used in AMBER, CHARMM, and OPLS force fields.
- **Hydrogen bonding**: The charge on the H-bond donor hydrogen (typically +0.3 to +0.5 $e$ from NPA) correlates strongly with H-bond strength.
- **Hyperconjugation in drug molecules**: NBO analysis reveals anomeric effects in carbohydrates, rotational barriers in amides (peptide bond), and conformational preferences in drug-like molecules.
- **Charge-transfer complexes**: The NPA charge transferred in donor–acceptor complexes quantifies the degree of charge-transfer character.

## 📋 Summary

| Method | Formula | Basis Sensitivity | Typical Use |
|--------|---------|:-----------------:|-------------|
| Mulliken | $(\mathbf{PS})_{\mu\mu}$ sum per atom | Very high ✗ | Quick checks, small basis |
| Löwdin | $(\mathbf{S}^{1/2}\mathbf{P}\mathbf{S}^{1/2})_{\mu\mu}$ | Moderate | Better than Mulliken |
| NPA (NBO) | NAO density per atom | Low ✓ | Publication quality |
| RESP | Fit to ESP grid | Low ✓ | Force field parametrisation |
| Wiberg BO | $\sum_{\mu\nu}(\mathbf{PS})_{\mu\nu}^2$ | Moderate | Bond multiplicity |

**Recommendation**: Always use NPA/NBO charges for publication; report Mulliken charges only for quick comparisons within the same basis set.

## 📝 Exercises

1. **Electronegativity trend**: Compute Mulliken and Löwdin charges on the central atom in CH₄, NH₃, H₂O, and HF at B3LYP/def2-SVP. Plot the central atom charge vs Pauling electronegativity (C: 2.55, N: 3.04, O: 3.44, F: 3.98). Is the correlation linear?

2. **Basis-set sensitivity**: Recompute the Mulliken charge on N in NH₃ using STO-3G, 3-21G, 6-31G*, def2-SVP, and def2-TZVP. How does the charge change? Is Löwdin more stable?

3. **Bond order vs bond length**: Compute Wiberg bond orders for N₂ (triple bond, 1.098 Å), N₂H₄ (single N–N, 1.449 Å), and N₂H₂ (double N=N, 1.252 Å). Do the computed bond orders follow the expected trend 3 > 2 > 1?

4. **Hyperconjugation in BF₃**: The B–F bond length in BF₃ (1.307 Å) is shorter than a typical B–F single bond (~1.39 Å), suggesting partial double-bond character due to F $p_z \to$ B $p_z$ back-donation. Compute the Wiberg B–F bond order and compare with BH₃ B–H bond order. Does the data support $\pi$ back-donation?

5. **Charge vs dipole moment**: Using the Mulliken charges from this notebook, estimate the dipole moment of H₂O as $\mu \approx \sum_i q_i \mathbf{r}_i$. Compare with the DFT dipole moment from `mf.dip_moment()`. Discuss why they differ.